# Task 2, Step 4: Stage 2, Visual Instruction Tuning

1,500 more updates with the same connector and newly enabled LoRA adapters: rank 16, alpha 32, dropout 0.05, on `q_proj`, `k_proj`, `v_proj`, `o_proj`. Original weights stay frozen. Connector peak LR $10^{-4}$, LoRA peak LR $2\times10^{-5}$.

Each draw: presence 50%, counting 25%, caption replay 25%.

**Image-swap penalty:** $\Delta\mathrm{NLL}=\mathrm{NLL}(y\mid q,I_{partner})-\mathrm{NLL}(y\mid q,I_{correct})$. Positive means the correct image makes the original reference more likely.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task2_vlm_grounding")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task2_vlm_grounding.src.notebook_utils import *
setup_notebook()

In [ ]:
from task2_vlm_grounding.src.notebook_utils import load_run
run = load_run()
history, dev, summary = run["history"], run["dev"], run["summary"]
scope = history[history["stage"] == 2]
if scope.empty:
    raise RuntimeError("Stage 2 has not been trained yet.")
print("Updates logged:", len(history), "of", summary["total_updates"], "| full recipe:", summary["full_recipe"])
print("Trainable parameters by stage:", summary["stage_params"], "| peak GPU MB:", round(summary["peak_gpu_mb"], 1))
print("Skipped updates:", len(summary["skipped_updates"]))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
axes[0].plot(scope["update"], scope["loss"], alpha=0.25, color="#3498db")
axes[0].plot(scope["update"], scope["loss"].rolling(25, min_periods=1).mean(), color="#2c3e50", label="25-update mean")
axes[0].set(title="Training loss (example-weighted)", xlabel="Main-run update", ylabel="Cross-entropy"); axes[0].legend()
axes[1].plot(scope["update"], scope["connector_lr"], label="Connector")
if scope["lora_lr"].notna().any():
    axes[1].plot(scope["update"], scope["lora_lr"], label="LoRA")
axes[1].set(title="Learning rates: 5% warmup, cosine to 10% of peak", xlabel="Main-run update", ylabel="LR", yscale="log"); axes[1].legend()
axes[2].plot(scope["update"], scope["grad_norm"], color="#e67e22", alpha=0.5)
axes[2].axhline(1.0, color="crimson", linestyle="--", label="Clip at 1.0")
axes[2].set(title="Gradient norm before clipping", xlabel="Main-run update", ylabel="Norm", yscale="log"); axes[2].legend()
save_figure("task2_stage2_training")

## 4.1 Mixture actually trained, and reference likelihood by image condition

In [ ]:
display(pd.DataFrame(summary["mixture_counts"]).fillna(0).astype(int))
stage_dev = dev[dev["update"].between(2250, 3750)]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for condition, color in [("correct", "#2ecc71"), ("swap", "#e74c3c"), ("no_image", "#95a5a6")]:
    axes[0].plot(stage_dev["update"], stage_dev[f"grounding_nll_{condition}"], marker="o", color=color, label=CONDITION_LABELS.get(condition))
axes[0].set(title="Grounding development: reference NLL", xlabel="Main-run update", ylabel="NLL"); axes[0].legend()
axes[1].plot(stage_dev["update"], stage_dev["grounding_swap_penalty"], marker="o", color="#8e44ad")
axes[1].axhline(0, color="black")
axes[1].set(title="Grounding image-swap penalty", xlabel="Main-run update", ylabel=r"$\Delta$NLL")
axes[2].plot(stage_dev["update"], stage_dev["caption_loss"], marker="o", color="#2980b9")
axes[2].set(title="Caption loss with replay", xlabel="Main-run update", ylabel="Cross-entropy")
save_figure("task2_stage2_development")